In [1]:
# ============================================================
# CELL 1
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import time

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


Mounted at /content/drive


In [2]:
# ============================================================
# CELL 2
# DATASET SETUP
# ============================================================

DATASET_ROOT = os.path.join(r"c:\Users\priya\Downloads\ok\TinyBayes-ML", "data", "dataset", "cotton")

SAVE_DIR = os.path.join(
    DATASET_ROOT,
    "run"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

RANDOM_STATE = 42

IMAGE_SIZE = 224

CLASS_NAMES = ['Bacterial_Blight', 'Curl_Virus', 'Fussarium_Wilt', 'Healthy', 'Leaf_Hopper_Jassids', 'Leaf_Redding', 'Leaf_Variegation']

print("="*60)
print("CLASSES")
print("="*60)

print(CLASS_NAMES)

# ------------------------------------------------------------
# Build dataframe from dataset directory
# ------------------------------------------------------------

data = []
for cls_name in CLASS_NAMES:
    cls_dir = os.path.join(DATASET_ROOT, cls_name)
    if os.path.exists(cls_dir):
        for img_file in os.listdir(cls_dir):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                data.append({'Image_ID': img_file, 'class': cls_name})

df = pd.DataFrame(data)

print()
print("Total Images :", len(df))
print()
print(df["class"].value_counts())

# ------------------------------------------------------------
# Encode labels
# ------------------------------------------------------------

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["class"])
NUM_CLASSES = len(label_encoder.classes_)

print()
print("Label Mapping")
for i, cls in enumerate(label_encoder.classes_):
    print(i, "->", cls)

# ------------------------------------------------------------
# Train / Validation Split (Stratified 80 / 20)
# ------------------------------------------------------------

df_train, df_val = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=RANDOM_STATE
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print()
print("="*60)
print("TRAIN SPLIT")
print("="*60)
print(df_train["class"].value_counts())

print()
print("="*60)
print("VALIDATION SPLIT")
print("="*60)
print(df_val["class"].value_counts())

# ------------------------------------------------------------
# Save splits
# ------------------------------------------------------------

df_train.to_csv(os.path.join(SAVE_DIR, "train_split.csv"), index=False)
df_val.to_csv(os.path.join(SAVE_DIR, "validation_split.csv"), index=False)

print()
print("Saved")
print(os.path.join(SAVE_DIR, "train_split.csv"))
print(os.path.join(SAVE_DIR, "validation_split.csv"))


CLASSES
['Bacterial_Blight', 'Curl_Virus', 'Fussarium_Wilt', 'Healthy', 'Leaf_Hopper_Jassids', 'Leaf_Redding', 'Leaf_Variegation']

Total Images : 3566

class
Curl_Virus             848
Bacterial_Blight       698
Healthy                682
Leaf_Redding           578
Fussarium_Wilt         419
Leaf_Hopper_Jassids    225
Leaf_Variegation       116

Label Mapping
0 -> Bacterial_Blight
1 -> Curl_Virus
2 -> Fussarium_Wilt
3 -> Healthy
4 -> Leaf_Hopper_Jassids
5 -> Leaf_Redding
6 -> Leaf_Variegation

TRAIN SPLIT
class
Curl_Virus             679
Bacterial_Blight       558
Healthy                545
Leaf_Redding           462
Fussarium_Wilt         335
Leaf_Hopper_Jassids    180
Leaf_Variegation        93

VALIDATION SPLIT
class
Curl_Virus             169
Bacterial_Blight       140
Healthy                137
Leaf_Redding           116
Fussarium_Wilt          84
Leaf_Hopper_Jassids     45
Leaf_Variegation        23

Saved
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cotton\run\train_sp

In [3]:
train_ids = set(df_train["Image_ID"])
val_ids = set(df_val["Image_ID"])

overlap = train_ids.intersection(val_ids)

print("Overlap:", len(overlap))


Overlap: 0


In [4]:
# ============================================================
# SAVE CLASS NAMES
# ============================================================

CLASS_NAMES_PATH = os.path.join(
    SAVE_DIR,
    "class_names.json"
)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(CLASS_NAMES, f, indent=4)

print("Class names saved to:")
print(CLASS_NAMES_PATH)

print()
print("Classes:")
print(CLASS_NAMES)


Class names saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cotton\run\class_names.json

Classes:
['Bacterial_Blight', 'Curl_Virus', 'Fussarium_Wilt', 'Healthy', 'Leaf_Hopper_Jassids', 'Leaf_Redding', 'Leaf_Variegation']


In [5]:
# ============================================================
# CELL 3
# EXTRACT MOBILENET FEATURES
# ============================================================

TRAIN_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "train_features.csv"
)

VAL_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "validation_features.csv"
)

if os.path.exists(TRAIN_FEATURE_PATH) and os.path.exists(VAL_FEATURE_PATH):

    print("="*60)
    print("FEATURE FILES FOUND")
    print("="*60)

    train_features_df = pd.read_csv(TRAIN_FEATURE_PATH)
    val_features_df = pd.read_csv(VAL_FEATURE_PATH)

    print(train_features_df.shape)
    print(val_features_df.shape)
    print()
    print(train_features_df.head())


FEATURE FILES FOUND
(2852, 578)
(714, 578)

      Image_ID           class    feat_0    feat_1    feat_2
0   fus320.jpg  Fussarium_Wilt -0.151609  0.339711  0.533205
1   curl52.jpg      Curl_Virus -0.080896 -0.106121  0.031344
2     h299.jpg         Healthy  0.190483 -0.069110  0.157017
3     h238.jpg         Healthy -0.102286  0.012859 -0.122100
4  curl418.png      Curl_Virus  0.136433 -0.072853 -0.097464

Train features shape: (2852, 578)
Val features shape:   (714, 578)


In [6]:
# ============================================================
# CELL 4
# TRAIN JACOBI-DMR
# ============================================================

import time

time_jacobi_start = time.time()

# ------------------------------------------------------------
# Separate features and labels
# ------------------------------------------------------------

feat_cols = [c for c in train_features_df.columns if c.startswith("feat_")]

X_train = train_features_df[feat_cols].values.astype(np.float32)
y_train = train_features_df["class"].values

X_val = val_features_df[feat_cols].values.astype(np.float32)
y_val = val_features_df["class"].values

# ------------------------------------------------------------
# One-hot encode targets
# ------------------------------------------------------------

y_train_df = pd.DataFrame({"c": y_train})
y_one_hot = pd.get_dummies(y_train_df["c"])

for cls in CLASS_NAMES:
    if cls not in y_one_hot.columns:
        y_one_hot[cls] = 0

y_one_hot = y_one_hot[CLASS_NAMES]

# ------------------------------------------------------------
# Jacobi regularized solve
# ------------------------------------------------------------

N = len(X_train)
a = 1.0 / N
b = 1.0 / N
k = 1.0

identity = np.eye(X_train.shape[1], dtype=np.float32)
regularization_lambda = 1.0

XtX = np.matmul(X_train.T, X_train)
XtX_reg = XtX + regularization_lambda * identity

jacobi_coefficients = {}

for cls in CLASS_NAMES:
    y_c = y_one_hot[cls].values
    eta = np.log((y_c + a) / (1.0 + k * b))
    Xty = np.matmul(X_train.T, eta)
    beta = np.linalg.solve(XtX_reg, Xty)
    jacobi_coefficients[cls] = beta

# ------------------------------------------------------------
# Validation inference
# ------------------------------------------------------------

preds_jacobi = []
for i in range(len(X_val)):
    x = X_val[i]
    scores = {cls: float(np.dot(x, jacobi_coefficients[cls])) for cls in CLASS_NAMES}
    preds_jacobi.append(max(scores, key=scores.get))

time_jacobi = time.time() - time_jacobi_start
acc_jacobi = accuracy_score(y_val, preds_jacobi)

print()
print("="*60)
print("JACOBI-DMR")
print("="*60)
print(f"Accuracy: {acc_jacobi:.4f}")
print(f"Time: {time_jacobi:.4f}s")
print()
print(classification_report(y_val, preds_jacobi, labels=CLASS_NAMES))
print()
print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_jacobi, labels=CLASS_NAMES))

# ------------------------------------------------------------
# Save coefficients
# ------------------------------------------------------------

JACOBI_JSON_PATH = os.path.join(SAVE_DIR, "jacobi_coefficients.json")

save_dict = {cls: [float(v) for v in jacobi_coefficients[cls]] for cls in CLASS_NAMES}

with open(JACOBI_JSON_PATH, "w") as f:
    json.dump(save_dict, f, indent=4)

print()
print("Jacobi coefficients saved to:")
print(JACOBI_JSON_PATH)



JACOBI-DMR
Accuracy: 0.9650
Time: 3.3040s

                     precision    recall  f1-score   support

   Bacterial_Blight       0.94      0.98      0.96       140
         Curl_Virus       0.97      0.98      0.98       169
     Fussarium_Wilt       0.98      1.00      0.99        84
            Healthy       0.97      0.96      0.97       137
Leaf_Hopper_Jassids       0.98      0.93      0.95        45
       Leaf_Redding       0.95      0.91      0.93       116
   Leaf_Variegation       1.00      1.00      1.00        23

           accuracy                           0.96       714
          macro avg       0.97      0.97      0.97       714
       weighted avg       0.97      0.96      0.96       714


Confusion Matrix:

[[137   1   0   1   0   1   0]
 [  0 166   0   1   0   2   0]
 [  0   0  84   0   0   0   0]
 [  1   2   2 132   0   0   0]
 [  0   0   0   1  42   2   0]
 [  7   2   0   1   1 105   0]
 [  0   0   0   0   0   0  23]]

Jacobi coefficients saved to:
c:\Users\priy

In [7]:
# ============================================================
# RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

import time

time_rf_start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

preds_rf = rf_model.predict(X_val)

time_rf = time.time() - time_rf_start

acc_rf = accuracy_score(y_val, preds_rf)

print()
print("="*60)
print("RANDOM FOREST")
print("="*60)

print(f"Accuracy: {acc_rf:.4f}")
print(f"Time: {time_rf:.4f}s")
print()

print(classification_report(y_val, preds_rf, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_rf, labels=CLASS_NAMES))



RANDOM FOREST
Accuracy: 0.9132
Time: 11.0121s

                     precision    recall  f1-score   support

   Bacterial_Blight       0.89      0.96      0.92       140
         Curl_Virus       0.91      0.98      0.94       169
     Fussarium_Wilt       0.93      0.98      0.95        84
            Healthy       0.91      0.95      0.93       137
Leaf_Hopper_Jassids       0.93      0.62      0.75        45
       Leaf_Redding       0.93      0.84      0.88       116
   Leaf_Variegation       1.00      0.65      0.79        23

           accuracy                           0.91       714
          macro avg       0.93      0.85      0.88       714
       weighted avg       0.92      0.91      0.91       714


Confusion Matrix:

[[135   3   0   1   0   1   0]
 [  1 165   0   1   0   2   0]
 [  0   2  82   0   0   0   0]
 [  1   3   3 130   0   0   0]
 [  2   5   0   8  28   2   0]
 [ 11   4   3   1   0  97   0]
 [  2   0   0   2   2   2  15]]


In [8]:
# ============================================================
# SUPPORT VECTOR MACHINE (SVM)
# ============================================================

from sklearn.svm import SVC

import time

time_svm_start = time.time()

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    random_state=42
)

svm_model.fit(X_train, y_train)

preds_svm = svm_model.predict(X_val)

time_svm = time.time() - time_svm_start

acc_svm = accuracy_score(y_val, preds_svm)

print()
print("="*60)
print("SUPPORT VECTOR MACHINE")
print("="*60)

print(f"Accuracy: {acc_svm:.4f}")
print(f"Time: {time_svm:.4f}s")
print()

print(classification_report(y_val, preds_svm, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_svm, labels=CLASS_NAMES))



SUPPORT VECTOR MACHINE
Accuracy: 0.9706
Time: 4.4160s

                     precision    recall  f1-score   support

   Bacterial_Blight       0.95      0.99      0.97       140
         Curl_Virus       0.99      0.99      0.99       169
     Fussarium_Wilt       0.97      1.00      0.98        84
            Healthy       0.98      0.98      0.98       137
Leaf_Hopper_Jassids       0.91      0.93      0.92        45
       Leaf_Redding       0.99      0.90      0.94       116
   Leaf_Variegation       1.00      1.00      1.00        23

           accuracy                           0.97       714
          macro avg       0.97      0.97      0.97       714
       weighted avg       0.97      0.97      0.97       714


Confusion Matrix:

[[138   1   0   1   0   0   0]
 [  0 168   0   1   0   0   0]
 [  0   0  84   0   0   0   0]
 [  0   0   3 134   0   0   0]
 [  1   0   0   1  42   1   0]
 [  7   1   0   0   4 104   0]
 [  0   0   0   0   0   0  23]]


In [9]:
# ============================================================
# RIDGE CLASSIFIER
# ============================================================

from sklearn.linear_model import RidgeClassifier

import time

time_ridge_start = time.time()

ridge_model = RidgeClassifier(
    alpha=1.0,
    random_state=42
)

ridge_model.fit(X_train, y_train)

preds_ridge = ridge_model.predict(X_val)

time_ridge = time.time() - time_ridge_start

acc_ridge = accuracy_score(y_val, preds_ridge)

print()
print("="*60)
print("RIDGE CLASSIFIER")
print("="*60)

print(f"Accuracy: {acc_ridge:.4f}")
print(f"Time: {time_ridge:.4f}s")
print()

print(classification_report(y_val, preds_ridge, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_ridge, labels=CLASS_NAMES))



RIDGE CLASSIFIER
Accuracy: 0.9664
Time: 0.1262s

                     precision    recall  f1-score   support

   Bacterial_Blight       0.95      0.98      0.96       140
         Curl_Virus       0.97      0.98      0.98       169
     Fussarium_Wilt       0.98      1.00      0.99        84
            Healthy       0.97      0.96      0.97       137
Leaf_Hopper_Jassids       0.98      0.93      0.95        45
       Leaf_Redding       0.95      0.91      0.93       116
   Leaf_Variegation       1.00      1.00      1.00        23

           accuracy                           0.97       714
          macro avg       0.97      0.97      0.97       714
       weighted avg       0.97      0.97      0.97       714


Confusion Matrix:

[[137   1   0   1   0   1   0]
 [  0 166   0   1   0   2   0]
 [  0   0  84   0   0   0   0]
 [  1   2   2 132   0   0   0]
 [  0   0   0   1  42   2   0]
 [  6   2   0   1   1 106   0]
 [  0   0   0   0   0   0  23]]


In [10]:
# ============================================================
# LOGISTIC REGRESSION (L1 / LASSO)
# ============================================================

from sklearn.linear_model import LogisticRegression

import time

time_lr_start = time.time()

lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

preds_lr = lr_model.predict(X_val)

time_lr = time.time() - time_lr_start

acc_lr = accuracy_score(y_val, preds_lr)

print()
print("="*60)
print("LOGISTIC REGRESSION (L1 / LASSO)")
print("="*60)

print(f"Accuracy: {acc_lr:.4f}")
print(f"Time: {time_lr:.4f}s")
print()

print(classification_report(y_val, preds_lr, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_lr, labels=CLASS_NAMES))



LOGISTIC REGRESSION (L1 / LASSO)
Accuracy: 0.9664
Time: 9.5146s

                     precision    recall  f1-score   support

   Bacterial_Blight       0.95      0.99      0.97       140
         Curl_Virus       0.97      0.97      0.97       169
     Fussarium_Wilt       0.98      1.00      0.99        84
            Healthy       0.99      0.97      0.98       137
Leaf_Hopper_Jassids       0.91      0.93      0.92        45
       Leaf_Redding       0.97      0.91      0.94       116
   Leaf_Variegation       0.96      1.00      0.98        23

           accuracy                           0.97       714
          macro avg       0.96      0.97      0.96       714
       weighted avg       0.97      0.97      0.97       714


Confusion Matrix:

[[139   1   0   0   0   0   0]
 [  0 164   1   0   2   2   0]
 [  0   0  84   0   0   0   0]
 [  1   2   1 133   0   0   0]
 [  0   1   0   1  42   1   0]
 [  6   1   0   1   2 105   1]
 [  0   0   0   0   0   0  23]]


In [11]:
# ============================================================
# FINAL MODEL COMPARISON
# ============================================================

comparison_df = pd.DataFrame({
    "Model": [
        "Support Vector Machine",
        "Ridge Classifier",
        "Logistic Regression (L1)",
        "Jacobi-DMR",
        "Random Forest"
    ],
    "Accuracy": [
        acc_svm,
        acc_ridge,
        acc_lr,
        acc_jacobi,
        acc_rf
    ],
    "Execution Time (s)": [
        time_svm,
        time_ridge,
        time_lr,
        time_jacobi,
        time_rf
    ]
})

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print()
print("="*70)
print("MODEL COMPARISON")
print("="*70)

print(comparison_df.to_string(index=False))

comparison_path = os.path.join(
    SAVE_DIR,
    "model_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print()
print(f"Comparison saved to:\n{comparison_path}")



MODEL COMPARISON
                   Model  Accuracy  Execution Time (s)
  Support Vector Machine    0.9706              4.4160
        Ridge Classifier    0.9664              0.1262
Logistic Regression (L1)    0.9664              9.5146
              Jacobi-DMR    0.9650              3.3040
           Random Forest    0.9132             11.0121

Comparison saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cotton\run\model_comparison.csv
